# 01 · Low-rank bilinear

The pipeline's linear model multiplies dimension *i* of the black card only with dimension *i* of the white card.
A bilinear score, `(P·black)·(Q·white)`, lets any direction of the black card pair with any direction of the white
card. The rank caps how many such pairings it can learn (the data is small).

Rank 0 keeps only the white-card term: card quality read from the text, the embedding version of popularity.
**If rank > 0 doesn't beat rank 0 on validation, the embeddings carry no usable black–white fit yet.**

In [ ]:
import os
import sys
from functools import partial
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "ml").is_dir() else Path.cwd().parent  # papermill: repo root; VS Code: notebooks/
sys.path.insert(0, str(ROOT))
os.environ.setdefault("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")

from torch import nn

from ml import experiments as ex
from ml import lower_priority, train
from ml.train import TrainConfig

print("CPU priority:", lower_priority())  # idle CPU only: the game comes first

In [ ]:
snapshot = None          # snapshot folder; None: the latest
embedding = "e5"         # minilm, e5 or granite
ranks = [0, 4, 16, 64]
center = True            # subtract the training mean vector (ex.center)
lr = 1e-2
weight_decay = 1e-3
max_epochs = 600
patience = 25
log_mlflow = True

In [ ]:
data = ex.load(snapshot)
print(data.snapshot.name, data.sizes())
inputs, targets = data.tensors(embedding)
if center:
    inputs = ex.center(inputs)
dim = inputs["train"][0].shape[-1]

In [ ]:
class Bilinear(nn.Module):
    """score = w·white + (P·black)·(Q·white); input (n, 1 + hand, dim), the black card first."""

    def __init__(self, dim, rank):
        super().__init__()
        self.white = nn.Linear(dim, 1)
        self.p = nn.Linear(dim, rank, bias=False) if rank else None
        self.q = nn.Linear(dim, rank, bias=False) if rank else None

    def forward(self, x):
        black, hand = x[:, :1], x[:, 1:]
        score = self.white(hand).squeeze(-1)
        if self.p is not None:
            score = score + (self.p(black) * self.q(hand)).sum(-1)
        return score

In [ ]:
config = TrainConfig(lr=lr, weight_decay=weight_decay, max_epochs=max_epochs, patience=patience)
results = ex.baselines(data)
# The pipeline's linear model on the same inputs, for reference.
results.append(ex.fit(f"pipeline linear {embedding}", partial(train.make_scorer, dim, train.CONFIGS["linear"]),
                      inputs, targets, train.CONFIGS["linear"],
                      {"model_type": "linear", "embedding": embedding, "center": center}))
for rank in ranks:
    results.append(ex.fit(f"bilinear r{rank} {embedding}", partial(Bilinear, dim, rank), inputs, targets, config,
                          {"model_type": "bilinear", "rank": rank, "embedding": embedding, "center": center}))
ex.table(results)

In [ ]:
if log_mlflow:
    print("MLflow parent run:", ex.log("01_bilinear", data, results, {"embedding": embedding, "center": center, "ranks": str(ranks)}))